In [20]:
# Loading all Neccessary Libraries
import pandas as pd
import numpy as np
from itertools import combinations
from numpy.linalg import solve
from scipy.stats import spearmanr  # optional: correlation check

# COnfiguring
SEASON = 2025   
RESULTS_PATH = "results.csv"        
RACES_PATH   = "races.csv"
CONSTRUCTORS_PATH = "constructors.csv"

# Mapping all the Power Units to the Constructors 
PU_MAPPING = {
    "Red Bull": "Honda",
#     "AlphaTauri": "Honda",
    "RB F1 Team": "Honda",

    "Mercedes": "Mercedes",
    "McLaren": "Mercedes",
    "Aston Martin": "Mercedes",
    "Williams": "Mercedes",

    "Ferrari": "Ferrari",
    "Haas F1 Team": "Ferrari",
    "Sauber": "Ferrari", 
#     "Alfa Romeo": "Ferrari",

    "Alpine F1 Team": "Renault"

}

# Loading all necessary files
results = pd.read_csv(RESULTS_PATH)    
races   = pd.read_csv(RACES_PATH)
constructors = pd.read_csv(CONSTRUCTORS_PATH)

# normalizing all text columns (trim)
for col in ["constructor_name", "name", "position", "status"]:
    if col in results.columns:
        results[col] = results[col].astype(str).str.strip()

# Mapping COnstructor Id to each constructor 
if "constructorId" in results.columns and "constructorId" in constructors.columns:
    # constructors likely has ['constructorId', 'name']
    results = results.merge(constructors[["constructorId","name"]], how="left", left_on="constructorId", right_on="constructorId")
    results = results.rename(columns={"name":"constructor"})
results["constructor"] = results["constructor"].astype(str).str.strip()

# === 3) Filtering the Season, by excluding all the sprints
season_races = races[races["year"] == SEASON]
season_race_ids = season_races["raceId"].unique()

# If results contain session type, filter; else filter by raceId list
results = results[results["raceId"].isin(season_race_ids)]

# Some datasets include sprint_results table, making sure to exclude sprint session rows if present
if "session" in results.columns:
    results = results[results["session"].str.lower() == "race"]

# === 4) Filtering so we keep only P1-P10.
# Convert position to numeric.
results["position"] = pd.to_numeric(results["position"], errors="coerce")
top10 = results[(results["position"].notna()) & (results["position"] <= 10)].copy()

# Sanity - each race should have more than 10 rows (some races may have <10 if penalties or missing)
print("unique races in top10:", top10["raceId"].nunique())

# === 5) MAP constructors to the power_unit according ot 
    # Use provided PU_MAPPING; ensure constructor names match keys exactly
top10["power_unit"] = top10["constructor"].map(PU_MAPPING)

# Check for unmapped constructors
unmapped = top10[top10["power_unit"].isna()]["constructor"].unique()
if len(unmapped) > 0:
    print("UNMAPPED constructors (fix PU_MAPPING):", unmapped)
    # Stop here so user fixes mapping
    raise SystemExit("Please update PU_MAPPING or constructors power-unit data.")

# === 6) Build per-race PU lists (only P1-P10) 
# For each race, building a DataFrame as raceId -> list of (position, pu)
race_groups = {}
for raceId, grp in top10.groupby("raceId"):
    # create sorted list by finishing position
    race_df = grp.sort_values("position")[["position","constructor","power_unit"]].copy()
    race_groups[raceId] = race_df.reset_index(drop=True)

print("Sample race:", list(race_groups.keys())[:3])

# === 7) Compute pairwise PU dominance across races ===

pus = sorted(top10["power_unit"].unique())
P = len(pus)
pu_to_idx = {pu: i for i, pu in enumerate(pus)}

# matrices
N = np.zeros((P, P), dtype=int)      # number of races where both PUs appear
score_for = np.zeros(P)
score_against = np.zeros(P)

def race_pair_margin(race_df, puA, puB):

    posA = race_df[race_df["power_unit"] == puA]["position"].values
    posB = race_df[race_df["power_unit"] == puB]["position"].values

    if len(posA) == 0 or len(posB) == 0:
        return None

    scoreA = 0
    scoreB = 0

    for pa in posA:
        for pb in posB:
            if pa < pb:
                scoreA += 1
            elif pb < pa:
                scoreB += 1

    return scoreA, scoreB


for raceId, race_df in race_groups.items():

    for puA, puB in combinations(pus, 2):

        result = race_pair_margin(race_df, puA, puB)

        if result is None:
            continue

        scoreA, scoreB = result

        i = pu_to_idx[puA]
        j = pu_to_idx[puB]

        # update comparison counts
        N[i, j] += 1
        N[j, i] += 1

        # accumulate dominance scores
        score_for[i] += scoreA
        score_against[i] += scoreB

        score_for[j] += scoreB
        score_against[j] += scoreA


# === 8) Build Colley matrix ===

games_played = N.sum(axis=1)

C = np.zeros((P, P))
b = np.zeros(P)

for i in range(P):
    C[i, i] = 2 + games_played[i]
    b[i] = 1 + (score_for[i] - score_against[i]) / 2

for i in range(P):
    for j in range(P):
        if i != j:
            C[i, j] = -N[i, j]

ratings = solve(C, b)

# === 9) Output results 
pu_table = pd.DataFrame({
    "power_unit": pus,
    "rating": ratings,
    "score_for": score_for,
    "score_against": score_against,
    "games": games_played
}).sort_values("rating", ascending=False).reset_index(drop=True)
pu_table.insert(0, "rank", pu_table.index + 1)
print(pu_table)

# === 10) Validation checks ===
print("PU list:", pus)
print("Games per PU (should be <= number of races):", games_played)
print("Total races considered:", len(race_groups))
# Optional - correlating with constructor points aggregated by PU

# First make a PU level points sum
if "points" in results.columns:
    pts = top10.groupby("power_unit")["points"].sum().reindex(pus).fillna(0).values
    corr, pval = spearmanr(ratings, pts)
    print("Spearman correlation between Colley rating and summed PU points:", corr, "p=", pval)

# For raceId in race_groups, log pairwise results to inspect particular races
sample_race = list(race_groups.keys())[0]
print("Sample raceId", sample_race)
races_df = pd.read_csv("races.csv")

race_name = races_df.loc[races_df["raceId"] == sample_race, "name"].values[0]
print("Race name:", race_name)

print(race_groups[sample_race])

unique races in top10: 19
Sample race: [1145, 1146, 1147]
   rank power_unit        rating  score_for  score_against  games
0     1   Mercedes  1.626984e+00      291.0          164.0     42
1     2      Honda  5.158730e-01      136.0          149.0     42
2     3    Renault -5.921189e-16        9.0           27.0     12
3     4    Ferrari -1.428571e-01      138.0          234.0     42
PU list: ['Ferrari', 'Honda', 'Mercedes', 'Renault']
Games per PU (should be <= number of races): [42 42 42 12]
Total races considered: 19
Spearman correlation between Colley rating and summed PU points: 0.39999999999999997 p= 0.6
Sample raceId 1145
Race name: Australian Grand Prix
   position   constructor power_unit
0         1       McLaren   Mercedes
1         2      Red Bull      Honda
2         3      Mercedes   Mercedes
3         4      Mercedes   Mercedes
4         5      Williams   Mercedes
5         6  Aston Martin   Mercedes
6         7        Sauber    Ferrari
7         8       Ferrari    Ferr

In [22]:
print("Score for:", score_for)
print("Score against:", score_against)
print("Total comparisons:", score_for.sum())

score_for.sum() == score_against.sum()

Score for: [138. 136. 291.   9.]
Score against: [234. 149. 164.  27.]
Total comparisons: 574.0


True